In [43]:
import numpy as np
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import hog

class StackedFaceRecognizer:
    def __init__(self, models_dict, train_feats, train_lbls):
        self.lbph = models_dict.get('LBPH')
        self.svm = models_dict.get('SVM')
        self.xgb = models_dict.get('XGB')
        self.le = models_dict.get('LE')  # Label Encoder for XGB
        self.X_train = np.array(train_feats)
        self.y_train = np.array(train_lbls)
        
        # Track IDs already predicted by each model in the current frame
        self.session_history = {
            'SVM': set(),
            'XGB': set(),
            'LBPH': set()
        }
        
    def reset_session(self):
        """THIS WAS MISSING: Clears history at the start of every frame."""
        self.session_history['SVM'].clear()
        self.session_history['XGB'].clear()
        self.session_history['LBPH'].clear()

    def extract_hog(self, image):
        return hog(image, orientations=9, pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

    def predict_cleaned(self, face_img, svm_pred, xgb_pred, lbph_pred):
        valid_votes = []
        
        # 1. Process SVM Vote
        if svm_pred:
            s_id = svm_pred['id']
            if s_id not in self.session_history['SVM']:
                valid_votes.append(s_id)
                self.session_history['SVM'].add(s_id)

        # 2. Process XGB Vote (Labels are already decoded in the GUI)
        if xgb_pred:
            x_id = xgb_pred['id']
            if x_id not in self.session_history['XGB']:
                valid_votes.append(x_id)
                self.session_history['XGB'].add(x_id)

        # 3. Process LBPH Vote
        if lbph_pred:
            l_id = lbph_pred['id']
            if l_id not in self.session_history['LBPH']:
                valid_votes.append(l_id)
                self.session_history['LBPH'].add(l_id)
        
        if not valid_votes: 
            return None, 0.0, "No Unique Votes"
        
        # Consensus logic
        top_vote, count = Counter(valid_votes).most_common(1)[0]
        
        if count >= 2:
            return top_vote, 1.0, f"Voting ({count}/3)"

        # Fallback: Restricted KNN using only candidates proposed by models
        valid_candidates = list(set(valid_votes))
        hog_vec = self.extract_hog(face_img)
        mask = np.isin(self.y_train, valid_candidates)
        
        if np.sum(mask) == 0: 
            return None, 0.0, "Err"
        
        n_neighbors = min(5, len(self.X_train[mask]))
        mini_knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='euclidean', algorithm='brute')
        mini_knn.fit(self.X_train[mask], self.y_train[mask])
        
        final_id = mini_knn.predict([hog_vec])[0]
        final_conf = np.max(mini_knn.predict_proba([hog_vec])[0])
        
        return final_id, final_conf, "Restricted KNN"

# IMPORTANT: Re-initialize the stacker object after running the class above
if 'models' in locals() and 'hog_feats' in locals():
    stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls)
    print("✅ Stacker updated with reset_session and XGB support.")



✅ Stacker updated with reset_session and XGB support.


In [44]:


# ==========================================
# 🛑 RUN THIS CELL FIRST (SETUP & LOADING)
# ==========================================
import joblib
import cv2
import xgboost as xgb
import numpy as np
import os
from skimage.feature import hog

# 1. Define the folder where images are (to get IDs)
DATA_PATH = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'

print("--- System Startup ---")

# --- A. Load Models ---
models = {}

try:
    models['SVM'] = joblib.load('svm_face_model.pkl')
    print("✅ SVM Loaded")
except: print("⚠️ SVM not found")

try:
    # Load XGBoost and its Label Encoder
    models['XGB'] = joblib.load('xgb_face_model.pkl')
    models['LE'] = joblib.load('label_encoder.pkl')
    print("✅ XGBoost Loaded")
except: print("⚠️ XGBoost or LabelEncoder not found")

try:
    lbph = cv2.face.LBPHFaceRecognizer_create()
    lbph.read('trainer.yml')
    models['LBPH'] = lbph
    print("✅ LBPH Loaded")
except: print("⚠️ LBPH not found")

# --- B. Create id_to_name Map ---
# This fixes the "name not defined" error. 
# It scans your folder and maps ID 1 -> "User 1", etc.
id_to_name = {}
if os.path.exists(DATA_PATH):
    img_files = os.listdir(DATA_PATH)
    unique_ids = set()
    for f in img_files:
        try:
            # Assumes format: User.1.jpg
            uid = int(f.split('.')[1])
            unique_ids.add(uid)
        except: pass
    
    # Map IDs to generic names (You can manually edit this later)
    for uid in unique_ids:
        id_to_name[uid] = f"User {uid}"
    print(f"✅ Generated names for {len(unique_ids)} users.")
else:
    print("⚠️ Dataset path not found. Names will be 'Unknown'.")

# --- C. Initialize Stacker ---
# We need to reload training data briefly to initialize the Stacker class
try:
    hog_feats, hog_lbls = joblib.load('hog_features.pkl')
    if 'StackedFaceRecognizer' in globals():
        stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls)
        print("✅ Stacker Initialized")
    else:
        print("❌ Error: Run the 'StackedFaceRecognizer' class cell first!")
except:
    print("⚠️ 'hog_features.pkl' missing. Stacker might fail.")

print("----------------------")
print("READY. Now run the GUI cell below.")


--- System Startup ---
✅ SVM Loaded
✅ XGBoost Loaded
✅ LBPH Loaded
✅ Generated names for 6 users.
✅ Stacker Initialized
----------------------
READY. Now run the GUI cell below.


In [ ]:
# =============================================================================
# 🛑 MASTER CELL: SETUP + NAMES + GUI (ALL IN ONE)
# =============================================================================
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage
import io
import copy
import os
import joblib
from collections import Counter
from skimage.feature import hog
import xgboost as xgb

print("--- 1. System Startup ---")

# ==========================================
# ✏️ YOUR MANUAL NAMES
# ==========================================
id_to_name = { 
    1: "Besheer", 
    2: "Ashraf", 
    3: "Seif", 
    4: "Sallam", 
    5: "Roger", 
    6: "Omar" 
}
print(f"✅ Loaded Names: {list(id_to_name.values())}")



# ==========================================
# 3. GUI WITH XGBOOST & RECOVERY
# ==========================================
print("\n--- 2. Launching GUI ---")

def compare_candidates(a, b):
    # 1. Primary: Highest Percentage
    if a['score'] > b['score']: return a
    if b['score'] > a['score']: return b
    
    # 2. Secondary: Highest XGB Score
    xgb_a = a['raw_stats'].get('XGB', 0)
    xgb_b = b['raw_stats'].get('XGB', 0)
    if xgb_a > xgb_b: return a
    if xgb_b > xgb_a: return b

    # 3. Tertiary: LBPH Quality
    lbph_a = a['raw_stats'].get('LBPH', 999)
    lbph_b = b['raw_stats'].get('LBPH', 999)
    a_good = lbph_a < 75
    b_good = lbph_b < 75

    if a_good and not b_good: return a
    if b_good and not a_good: return b
    if a_good and b_good: return a if lbph_a < lbph_b else b

    # 4. Final: SVM
    svm_a = a['raw_stats'].get('SVM', 0)
    svm_b = b['raw_stats'].get('SVM', 0)
    return a if svm_a > svm_b else b

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

model_selector = widgets.ToggleButtons(
    options=['LBPH', 'SVM', 'XGB', 'Stacked'],
    description='Model:',
    button_style='success'
)
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_disp = widgets.Output()
out_table = widgets.Output()

def process_image(change):
    out_disp.clear_output(); out_table.clear_output()
    if not uploader.value: return
    try:
        if isinstance(uploader.value, tuple): f = uploader.value[0]
        else: f = next(iter(uploader.value.values()))
        img_pil = PILImage.open(io.BytesIO(f['content']))
        img_np = np.array(img_pil)
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np
    except: return

    detect_neighbors = 3 if model_selector.value != 'LBPH' else 4
    faces_rects = face_cascade.detectMultiScale(gray, 1.06, detect_neighbors, minSize=(49, 49))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    all_faces = []

    # --- STEP 1: CALCULATE RAW PREDICTIONS ---
    for i, (x, y, w, h) in enumerate(faces_rects):
        roi = cv2.resize(gray[y:y+h, x:x+w], (200, 200), interpolation=cv2.INTER_CUBIC)
        face_final = clahe.apply(cv2.bilateralFilter(roi, 5, 75, 75))
        
        preds = {'SVM': None, 'XGB': None, 'LBPH': None}
        raw_stats = {'SVM': 0.0, 'XGB': 0.0, 'LBPH': 999.0}
        
        # SVM
        if 'SVM' in models:
            vec = stacker.extract_hog(face_final) if stacker else None
            try:
                prob = models['SVM'].predict_proba([vec])[0]
                conf = np.max(prob)
                raw_stats['SVM'] = conf
                if conf > 0.35:
                    preds['SVM'] = {'id': models['SVM'].classes_[np.argmax(prob)], 'conf': conf}
            except: pass
        
        # XGB (Replaces KNN)
        if 'XGB' in models and 'LE' in models:
            vec = stacker.extract_hog(face_final) if stacker else None
            try:
                prob = models['XGB'].predict_proba([vec])[0]
                conf = np.max(prob)
                raw_stats['XGB'] = conf
                if conf > 0.50:
                    class_idx = np.argmax(prob)
                    decoded_id = models['LE'].inverse_transform([class_idx])[0]
                    preds['XGB'] = {'id': decoded_id, 'conf': conf}
            except: pass

        # LBPH
        if 'LBPH' in models:
            try:
                pid, dist = models['LBPH'].predict(face_final)
                raw_stats['LBPH'] = dist
                if dist < 80:
                    preds['LBPH'] = {'id': pid, 'conf': dist}
            except: pass
        
        all_faces.append({
            'roi': face_final, 'bbox': (x,y,w,h), 
            'preds': preds, 
            'unfiltered': copy.deepcopy(preds),
            'raw_stats': raw_stats, 'img_roi': img_np[y:y+h, x:x+w]
        })

    # --- STEP 2: PRE-STACK FILTERING ---
    for model_name in ['SVM', 'XGB', 'LBPH']:
        candidates = []
        for idx, face in enumerate(all_faces):
            p = face['preds'][model_name]
            if p: candidates.append((idx, p['id'], p['conf']))
        
        grouped = {}
        for c in candidates:
            if c[1] not in grouped: grouped[c[1]] = []
            grouped[c[1]].append(c)
            
        for pid, items in grouped.items():
            if len(items) > 1:
                is_high_best = (model_name != 'LBPH')
                items.sort(key=lambda x: x[2], reverse=is_high_best)
                for loser in items[1:]:
                    all_faces[loser[0]]['preds'][model_name] = None

    # --- STEP 3: STACKING ---
    final_candidates = []
    if stacker: stacker.reset_session()

    for face in all_faces:
        name = "Unknown"
        score_val = 0.0
        score_txt = ""
        info = ""
        
        try:
            if model_selector.value == 'Stacked' and stacker:
                pid, conf, method = stacker.predict_cleaned(
                    face['roi'], face['preds']['SVM'], face['preds']['XGB'], face['preds']['LBPH']
                )
                score_val = float(conf)
                if pid: name = id_to_name.get(pid, "Unknown")
                score_txt = f"{int(score_val*100)}%"
                info = method

            elif model_selector.value in ['SVM', 'XGB', 'LBPH']:
                m = model_selector.value
                p = face['preds'][m]
                if p:
                    name = id_to_name.get(p['id'], "Unknown")
                    score_val = p['conf']
                    score_txt = f"{p['conf']*100:.1f}%" if m!='LBPH' else f"{p['conf']:.1f}"
                    info = m
                else: info = "Filtered"
        except Exception as e: info = "Err"
        
        final_candidates.append({
            'bbox': face['bbox'], 'name': name, 'score': score_val, 
            'score_txt': score_txt, 'info': info, 'roi': face['img_roi'],
            'raw_stats': face['raw_stats'],
            'raw_preds': face['unfiltered'] 
        })

    # --- STEP 4: DEDUPLICATION ---
    grouped_final = {}
    for i, res in enumerate(final_candidates):
        if res['name'] == "Unknown": continue
        if res['name'] not in grouped_final: grouped_final[res['name']] = []
        grouped_final[res['name']].append(i)
    
    active_names = set()
    losers_indices = []
    
    for name, indices in grouped_final.items():
        if len(indices) == 1:
            active_names.add(name)
        else:
            winner_idx = indices[0]
            for challenger_idx in indices[1:]:
                best = compare_candidates(final_candidates[winner_idx], final_candidates[challenger_idx])
                if best is final_candidates[challenger_idx]: winner_idx = challenger_idx
            active_names.add(name)
            for idx in indices:
                if idx != winner_idx: losers_indices.append(idx)

    # --- RECOVERY ---
    # --- RECOVERY ---
    for idx in losers_indices:
        loser = final_candidates[idx]
        preds = loser['raw_preds']
        recovered = False
        
        # 1. Try LBPH (First Priority)
        lbph_p = preds.get('LBPH')
        if lbph_p and lbph_p['conf'] < 75:
            rec_name = id_to_name.get(lbph_p['id'], "Unknown")
            if rec_name not in active_names:
                final_candidates[idx]['name'] = rec_name
                final_candidates[idx]['info'] = "Recovered (LBPH)"
                final_candidates[idx]['score_txt'] = f"{lbph_p['conf']:.1f}"
                active_names.add(rec_name); recovered = True

        # 2. Try SVM (Added Here)
        if not recovered:
            svm_p = preds.get('SVM')
            if svm_p:
                rec_name = id_to_name.get(svm_p['id'], "Unknown")
                if rec_name not in active_names:
                    final_candidates[idx]['name'] = rec_name
                    final_candidates[idx]['info'] = "Recovered (SVM)"
                    final_candidates[idx]['score_txt'] = f"{svm_p['conf']*100:.1f}%"
                    active_names.add(rec_name); recovered = True
        
        # 3. Try XGB (Last Priority)
        if not recovered:
            xgb_p = preds.get('XGB')
            if xgb_p:
                rec_name = id_to_name.get(xgb_p['id'], "Unknown")
                if rec_name not in active_names:
                    final_candidates[idx]['name'] = rec_name
                    final_candidates[idx]['info'] = "Recovered (XGB)"
                    final_candidates[idx]['score_txt'] = f"{xgb_p['conf']*100:.1f}%"
                    active_names.add(rec_name); recovered = True

        # 4. Give Up
        if not recovered:
            final_candidates[idx]['name'] = "Unknown"
            final_candidates[idx]['info'] = "Duplicate"
            final_candidates[idx]['score_txt'] = ""

    # --- DISPLAY ---
    with out_disp:
        img_disp = img_np.copy()
        if not final_candidates: print("No faces found.")
        for res in final_candidates:
            x, y, w, h = res['bbox']
            color = (255, 0, 0) if res['name'] == "Unknown" else (0, 255, 0)
            label = f"{res['name']} ({res['score_txt']})" if res['name'] != "Unknown" else "Unknown"
            cv2.rectangle(img_disp, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img_disp, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        display(PILImage.fromarray(img_disp))

    with out_table:
        rows = []
        for res in final_candidates:
            thumb = cv2.imencode('.png', cv2.cvtColor(res['roi'], cv2.COLOR_RGB2BGR))[1].tobytes()
            style = "color:green" if res['name'] != "Unknown" else "color:red"
            html = f"<b style='{style}'>{res['name']}</b><br><i>{res['info']}</i> {res['score_txt']}"
            rows.append(widgets.HBox([widgets.Image(value=thumb, format='png', width=50), widgets.HTML(html)]))
        display(widgets.VBox(rows))

uploader.observe(process_image, names='value')
model_selector.observe(process_image, names='value')
display(widgets.VBox([widgets.HTML("<h3>Face System v19 (Named + XGB)</h3>"), model_selector, uploader, out_disp, out_table]))

--- 1. System Startup ---
✅ Loaded Names: ['Besheer', 'Ashraf', 'Seif', 'Sallam', 'Roger', 'Omar']

--- 2. Launching GUI ---
